# colab CoNGA pipeline

This notebook will install the required packages, download the CoNGA package and some example datasets, and run a few analyses. You can save a copy of it and edit and run it. If you want to test CoNGA without going to the trouble of installing locally, you can upload your datafiles to a colab instance and run conga on them by changing the file locations in the 'Running' section below. Upload files using the 'file explorer' tab on the left.

# Installation

## Install `scanpy` (for single-cell) and `leiden` (for clustering)

`colab` comes with matplotlib, pandas, numpy, scipy, and a few others pre-installed. So you would probably need a few others in a local install into a clean environment. See the instructions in the CoNGA github readme.

If you are running this notebook locally, not in colab, note that these commmands will change whichever python environment was used to launch the notebook. And they also won't work if that environment doesn't match the kernel of this notebook. See the nice explanations here:

https://jakevdp.github.io/blog/2017/12/05/installing-python-packages-from-jupyter/

You could replace !pip with %pip if your notebook understands pip magic and that should help to sync up the two install locations...


In [ ]:
!pip install leidenalg scanpy 


## Install a tool for converting from SVG to PNG files

In [ ]:
!pip install cairosvg

## download the `conga` repository 

In [ ]:
!git clone https://github.com/phbradley/conga.git

## compile the C++ implementation of TCRdist
This is optional for the core graph vs graph pipeline but will speed things up for large datasets and is required for tcr_clumping or tcr database matching

In [ ]:
!cd conga/tcrdist_cpp/ ; make


## add `conga` to our path so we can `import` it
The `conga/` github repository folder contains within it the `conga` package folder, which is what we need to be able to find in our path when we are trying to `import conga`.  

In [ ]:
import sys
sys.path.append('conga/')

## Download some example 10x datasets. 
See the README file contained in the download folder for license information. 

In [ ]:
!wget https://www.dropbox.com/s/r7rpsftbtxl89y5/conga_example_datasets_v1.zip
!unzip conga_example_datasets_v1.zip

In [ ]:
!cat conga_example_datasets_v1/README.md

In [ ]:
!ls conga_example_datasets_v1/

# Running

In [ ]:
%matplotlib inline
import conga
import numpy as np
import scanpy as sc
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image               # for looking at some of 
from IPython.display import display # the output images.


## Input data, output paths
You could upload your own data to colab and analyze it. Just change these filenames to point to the correct locations.

In [ ]:
# you might have to change these paths depending on what you want to analyze
gex_datafile = 'conga_example_datasets_v1/vdj_v1_hs_pbmc3_5gex_filtered_gene_bc_matrices_h5.h5'
gex_datatype = '10x_h5' # other possibilities right now: ['10x_mtx', 'h5ad'] (h5ad from scanpy)
tcr_datafile = 'conga_example_datasets_v1/vdj_v1_hs_pbmc3_t_filtered_contig_annotations.csv'
organism = 'human'

clones_file = 'vdj_v1_hs_pbmc3_clones.tsv'
outfile_prefix = 'hs_pbmc3_test1'


*Everything below here is identical to the corresponding sections of simple_conga_pipeline.ipynb*

## Setup for conga: make a TCRdist clones_file

In [ ]:
# this creates the TCRdist 'clones file'
conga.tcrdist.make_10x_clones_file.make_10x_clones_file( tcr_datafile, organism, clones_file )

# Note: we no longer precompute a KernelPCA kernel-PCs file here. CoNGA automatically
# selects the best available TCR representation when the data is loaded below --
# the fast vectorized TCRdist encoding for organisms in
# conga.tcrdist.vectorized.SUPPORTED_ORGANISMS (18 organisms as of this writing,
# including 'human'), falling back to KernelPCA or exact TCRdist neighbors otherwise.

## Read the data, create a `scanpy` `AnnData` object with everything inside

In [ ]:
adata = conga.preprocess.read_dataset(
    gex_datafile, gex_datatype, clones_file, allow_missing_kpca_file=True,
)

# store the organism info in adata
adata.uns['organism'] = organism

adata

Note that the AnnData object contains the TCR sequence information in the `obs` array, stored with the  `['va',..., 'cdr3b_nucseq']` fields.

The TCR neighbor representation (vectorized encoding, KernelPCA, or exact TCRdist, depending on organism and dataset size) is selected and built automatically during the clustering/dimensionality-reduction step below, not at this point.

The specific representation array isn't guaranteed to exist yet at this point -- it depends on which representation gets selected automatically later. For organisms in `conga.tcrdist.vectorized.SUPPORTED_ORGANISMS` (including `'human'`), it will be `adata.obsm['X_vec_tcr']`; otherwise it will be `adata.obsm['X_pca_tcr']` (KernelPCA). Run `conga.preprocess.cluster_and_tsne_and_umap(adata)` below first, then check `conga.preprocess.get_active_tcr_representation(adata)` to see which one was actually used.

In [ ]:
# CDR3-alpha regions:
adata.obs['cdr3a'].head(3)

## Do some very basic scGEX filtering

You could customize this or replace it with something else entirely.

After running this (or your replacement), the following should be true:
* `adata.X` is a dense matrix just containing the highly variable genes, normalized, `log1p`ed, scaled, and ready for running PCA.
* The T cell receptor V and J genes have been eliminated from `adata.X`, so they don't influence the GEX clusters/neighbor graphs. (Or the IG receptor genes, if analyzing BCR data).
* 'Antibody' features (ADT/surface protein/pMHC) have been eliminated from `adata.X`. See the `run_conga.py` argument `--include_protein_features` for an approach to including these additional features into the GEX graphs. They can still be present in `adata.raw.X`; if so they will be included in DEG analyses. 
* `adata.raw.X` has been normalized (all rows sum to 10,000) and `log1p`'ed
* Cells with too few or too many genes, or high mitochondrial expression, have been eliminated.

It wouldn't be a bad idea to explicitly look for and remove doublet cells, too. That is not done automatically inside this function.

In [ ]:
adata = conga.preprocess.filter_and_scale( 
    adata, 
    min_genes_per_cell=200,
    max_genes_per_cell=2500,
    max_percent_mito=0.1,
)

## Now we reduce to a single cell per TCR clonotype

Do this by computing GEX distances and picking the most representative cell for each clonotype (the one with the smallest average distance to all other cells in the clonotype)

In [ ]:
adata = conga.preprocess.reduce_to_single_cell_per_clone(adata)

adata

## Now that we've reduced to a single cell per clonotype, run clustering and dimensionality reduction for GEX and for TCR

In [ ]:
adata = conga.preprocess.cluster_and_tsne_and_umap( adata )


### These features are stored in `adata.obs` and in `adata.obsm`

* `adata.obs['clusters_gex']` -- GEX clusters
* `adata.obs['clusters_tcr']` -- TCR clusters
* `adata.obsm['X_gex_2d']` -- GEX 2D landscape coordinates (from UMAP)
* `adata.obsm['X_tcr_2d']` -- TCR 2D landscape coordinates (from UMAP)
* `adata.obsm['X_pca_gex']` -- GEX principal components
* `adata.obsm['X_pca_tcr']` -- TCRdist kernel principal components (only present if KernelPCA was the selected representation; use `conga.preprocess.get_active_tcr_representation(adata)` to check which representation was actually used)


In [ ]:
adata

### Make UMAP landscape plots colored by cluster assignments

In [ ]:
plt.figure(figsize=(12,6))
plt.subplot(121)
xy = adata.obsm['X_gex_2d']
clusters = np.array(adata.obs['clusters_gex'])
cmap = plt.get_cmap('tab20')
colors = [ cmap.colors[x] for x in clusters]
plt.scatter( xy[:,0], xy[:,1], c=colors)
plt.title('GEX UMAP colored by GEX clusters')

plt.subplot(122)
xy = adata.obsm['X_tcr_2d']
clusters = np.array(adata.obs['clusters_tcr'])
cmap = plt.get_cmap('tab20')
colors = [ cmap.colors[x] for x in clusters]
plt.scatter( xy[:,0], xy[:,1], c=colors)
plt.title('TCR UMAP colored by TCR clusters');



## Compute the GEX and TCR neighbor sets

The neighbors are stored as GEX,TCR tuples in the `all_nbrs` dictionary indexed by `nbr_frac` (neighbor fraction: nbr_frac = 0.1 means K-nearest neighbor graph with `K = 0.1*num_clonotypes`)

So, to unpack the GEX and TCR neighbor sets for a given value of `nbr_frac`, do: 
```
nbrs_gex, nbrs_tcr = all_nbrs[nbr_frac]
```

We also do a bit of other miscellaneous setup here...

In [ ]:
# these are the nbrhood sizes, as a fraction of the entire dataset:
nbr_fracs = [0.01, 0.1]

# we use this nbrhood size for computing the nndists
nbr_frac_for_nndists = 0.01

all_nbrs, nndists_gex, nndists_tcr = conga.preprocess.calc_nbrs(
    adata, nbr_fracs, also_calc_nndists=True, nbr_frac_for_nndists=nbr_frac_for_nndists)

# stash these in obs array, they are used in a few places...
adata.obs['nndists_gex'] = nndists_gex
adata.obs['nndists_tcr'] = nndists_tcr

conga.preprocess.setup_tcr_cluster_names(adata) #stores in adata.uns

## Run graph-vs-graph analysis

This will also save the results (a pandas `DataFrame`) to a tsv file `<outfile_prefix>_graph_vs_graph.tsv` and store them in the `adata.uns['conga_results']` dictionary.

In [ ]:
results = conga.correlations.run_graph_vs_graph(
    adata, all_nbrs, outfile_prefix=outfile_prefix)

results.head()

### conga results are stored in `adata.uns['conga_results']`

This is a dictionary whose keys are strings that can be found in `conga/tags.py` 

Each result should have a corresponding help message (under the key with a `'_help'` suffix) 

In [ ]:
adata.uns['conga_results'].keys()

In [ ]:
# another way of getting the graph-vs-graph results DataFrame:
# conga.tags.GRAPH_VS_GRAPH = 'graph_vs_graph'
results = adata.uns['conga_results'][conga.tags.GRAPH_VS_GRAPH]
results.head() 

In [ ]:
print(adata.uns['conga_results']['graph_vs_graph_help'])

### Some potentially useful statistics are stored in `adata.uns['conga_stats']`

In [ ]:
adata.uns['conga_stats']

### Make a scatterplot colored by conga score

In [ ]:
#put the conga hits on top
conga_scores = adata.obs['conga_scores']
colors = np.sqrt(np.maximum(-1*np.log10(conga_scores),0.0))
reorder = np.argsort(colors)

plt.figure(figsize=(12,6))
plt.subplot(121)
xy = adata.obsm['X_gex_2d']
plt.scatter( xy[reorder,0], xy[reorder,1], c=colors[reorder], vmin=0, vmax=np.sqrt(5))
plt.title('GEX UMAP colored by conga score')

plt.subplot(122)
xy = adata.obsm['X_tcr_2d']
plt.scatter( xy[reorder,0], xy[reorder,1], c=colors[reorder], vmin=0, vmax=np.sqrt(5))
plt.title('TCR UMAP colored by conga score');


## Make the CoNGA cluster logos plot

This plot summarizes the CoNGA clusters of size at least 5 (`min_cluster_size`). Some useful options:

* `gex_header_genes`: list of gene names for the thumbnail UMAP plots above the logos. Default is `conga.plotting.default_gex_header_genes[organism]`
* `gex_header_tcr_score_names`: list of tcr scores to show in the thumbnail UMAP plots. Default is `['imhc', 'cdr3len', 'cd8', 'nndists_tcr']`
* `logo_genes`: python list of gene names for the 'GEX logo' shown on the right. Should have length = `logo_gene_width*3 - 2`. Default is `conga.plotting.default_logo_genes[organism]`
* `logo_gene_width`: width of the 'GEX logo' (default is 6).

Note: These arguments also work for the `conga.plotting.make_tcr_clumping_plots` function below.

In [ ]:
nbrs_gex, nbrs_tcr = all_nbrs[0.1]

min_cluster_size = 5

conga.plotting.make_graph_vs_graph_logos(
    adata,
    outfile_prefix,
    min_cluster_size,
    nbrs_gex,
    nbrs_tcr,
)

# or equivalently:
#
# conga.plotting.make_graph_vs_graph_logos(
#     adata,
#     outfile_prefix,
#     min_cluster_size,
#     nbrs_gex,
#     nbrs_tcr,
#     gex_header_genes = ['clone_sizes','CD4','CD8A','CD8B','SELL','GNLY','GZMA','CCL5','ZNF683','IKZF2','PDCD1','KLRB1'],
#     gex_header_tcr_score_names = ['imhc', 'cdr3len', 'cd8', 'nndists_tcr'],
#     logo_genes = ['CD4','CD8A','CD8B','CCR7','SELL','GNLY','PRF1','GZMA','IL7R','IKZF2','KLRD1','CCL5','ZNF683','KLRB1','NKG7','HLA-DRB1' ],
#     gene_logo_width = 6,
# )



### The image that was saved to disk is higher resolution than the one above here in the notebook 
If you have the python image library (PIL) installed, the following commands should display it.

The image filenames are stored in `adata.uns['conga_results']`. This comes in handy for the html-summary generating code (see the end of this notebook). 

In [ ]:
from PIL import Image               # for looking at some of 
from IPython.display import display # the output images.

tag = conga.tags.GRAPH_VS_GRAPH_LOGOS
pngfile = adata.uns['conga_results'][tag]
help_message = adata.uns['conga_results'][tag+'_help']

print(help_message)
image = Image.open(pngfile)
display(image)


## Run graph-vs-feature analysis, comparing the TCR graph to GEX features (mostly gene expression) and the GEX graph to TCR features (CDR3 features and V/J gene segments)

### First run the analysis, stashing the results in `adata.uns['conga_results']` and saving tables as `.tsv` files (since we pass in `outfile_prefix`)

In [ ]:
conga.correlations.run_graph_vs_features(
    adata, all_nbrs, outfile_prefix=outfile_prefix)



### After running the analysis we can make the graph-vs-features plots

In [ ]:
conga.plotting.make_graph_vs_features_plots(
    adata, all_nbrs, outfile_prefix)

## TCR matching to database

Here we match the TCRs in the dataset to a database of paired TCR sequences of 
known epitope specificity (or to a user-provided database using the `db_tcrs_tsvfile`
argument). The default database is `conga/data/new_paired_tcr_db_for_matching_nr.tsv` (see the readme file `conga/data/new_paired_tcr_db_for_matching_nr_README.txt` for information about where these TCR sequences come from. Major contributors are the VDJdb database, the McPAS database, the 10x 200k dataset, the Dash et al TCRdist dataset.)

In [ ]:
match_results = conga.tcr_clumping.match_adata_tcrs_to_db_tcrs(
    adata, num_random_samples_for_bg_freqs=50000)

match_results.head()

## TCR clumping

This analysis identifies TCR neighborhoods that are overpopulated relative to a background
expectation based on a simple model of V(D)J recombination. These TCR clumps or clusters may
represent epitope-specific responses, invariant T cell populations, or other potentially
interesting T cell subsets. This analysis does not directly use the GEX information,
although we can subsequently look at GEX features of the identified clusters (see the tcr clumping logo plots below)



In [ ]:
results = conga.tcr_clumping.assess_tcr_clumping(
    adata,
    outfile_prefix= outfile_prefix, # will save results as .tsv file
)

results.head()

### Make the tcr clumping plots

We can pass the same logo-configuring arguments to this function as to
`make_graph_vs_graph_logos` (see the graph-vs-graph description up above).

In [ ]:
nbrs_gex, nbrs_tcr = all_nbrs[ max(nbr_fracs) ]

# now call plotting function, after results are stashed in adata
conga.plotting.make_tcr_clumping_plots(
    adata,
    nbrs_gex,
    nbrs_tcr,
    outfile_prefix,
    )


## Graph-vs-features analysis using the Yosef lab's `HotSpot` algorithm

DeTomaso, D., & Yosef, N. (2021). Hotspot identifies informative gene modules across modalities of single-cell genomics. Cell Systems, 12(5), 446–456.e9.

Here we use the HotSpot algorithm to identify TCR features that correlate with the GEX neighbor graph, or GEX features (expression of individual genes) that correlate with the TCR neighbor graph. The basic idea is that if two clonotypes are neighbors in the graph, then their feature values are correlated. This can be more sensitive than our original graph-vs-features analysis, which looks at the feature score distribution in each clonotype's
neighborhood individually, rather than across the entire graph (using the HotSpot 
H-statistic). On the other hand, features that are elevated in a very smal subpopulation 
may get a more significant score in the original CoNGA graph-vs-features analysis. So it's
worth looking at both analyses. 

This is a 'beta' version that does not incorporate all the cleverness in the HotSpot
algorithm. We find that neighbor graphs with small neighborhoods (e.g., nbr_fracs < 0.02)
sometimes give less interpretable/robust results. So here we are using only the 0.1
nbr_frac (ie, the K nearest neighbor graph with K = 0.1 * num_clonotypes).



In [ ]:
results = conga.correlations.find_hotspots_wrapper(
    adata, all_nbrs, nbr_fracs = [0.1], outfile_prefix=outfile_prefix)

results.head(10)

In [ ]:
conga.plotting.make_hotspot_plots(
    adata, all_nbrs, outfile_prefix,
    )

## CoNGA can generate a summary `.html` file using the results in `adata.uns['conga_results']`

In [ ]:

html_file = outfile_prefix+'_results_summary.html'
conga.plotting.make_html_summary(adata, html_file)

### display the summary html output here in the notebook

In [ ]:
from IPython.display import display, HTML

html_lines = open(html_file,'r').read()

display(HTML(html_lines))